# 06 — Split sạch, retrain & feedback loop (Giai đoạn 12)

Spec: `docs/specs/g12-sua-ro-ri-va-do-prod.md`.

Split Roboflow gốc bị rò rỉ: 59/101 ảnh test và 122/203 ảnh valid là bản sao
(lật ngang / crop) của ảnh train. Notebook này:

1. **T12.2** dựng lại split sạch từ manifest đã commit
   (`data/splits/yoga_v1_clean_manifest.csv`) — 0 nhóm ảnh trùng cắt ngang split.
2. **T12.3** đo model cũ (`yolov8n_v1_baseline`) chỉ trên 42 ảnh test gốc *sạch*.
3. **T12.5** train lại baseline trên split sạch → `yolov8n_v1_clean_baseline`.
4. **T12.6** error analysis trên lỗi thật (test mới + OOD), EigenCAM trên ảnh **sai**.
5. **T12.7** một vòng cải tiến → `yolov8n_v1_clean_feedback`, bảng trước/sau.

**Chạy trên Colab** (cần GPU — Runtime → Change runtime type → T4 GPU).

## Setup — mount Drive + cd vào repo

Giống các notebook trước. Cần `data/raw/yoga_v1/` (tải ở Giai đoạn 1).

In [ ]:
import os

REPO_DIR_NAME = "computer-vision-project"  # đổi nếu bạn git clone ra tên thư mục khác

try:
    from google.colab import drive

    drive.mount("/content/drive")
    drive_path = f"/content/drive/MyDrive/{REPO_DIR_NAME}"
    if not os.path.isdir(drive_path):
        raise FileNotFoundError(
            f"{drive_path} không tồn tại — kiểm tra lại bạn đã `git clone` repo vào "
            "đúng chỗ trong Drive chưa (T0.4), hoặc sửa REPO_DIR_NAME ở trên cho khớp."
        )
    os.chdir(drive_path)
except ImportError:
    pass  # không chạy trên Colab (vd Jupyter local) — giả định cwd đã là repo root

print("cwd:", os.getcwd())
assert os.path.isdir("scripts") and os.path.isdir("data"), (
    "Chưa đứng ở repo root — không thấy scripts/ và data/ ở cwd hiện tại."
)
assert os.path.isfile("data/raw/yoga_v1/data.yaml"), (
    "data/raw/yoga_v1/data.yaml chưa có — chạy notebooks/01_data_exploration.ipynb "
    "(Giai đoạn 1) trước để tải dataset."
)

## Đồng bộ code mới nhất

In [ ]:
assert os.path.isdir(".git"), (
    "cwd hiện tại không phải repo root (không thấy .git) — runtime Colab có "
    "thể vừa bị reset. Chạy lại cell 'Setup — mount Drive + cd vào repo' ở "
    "trên (mount + cd) trước, rồi chạy lại cell này."
)
!git pull

## Cài dependencies

In [ ]:
!pip install -q -r requirements.txt
import torch
import ultralytics
from pytorch_grad_cam import EigenCAM  # noqa: F401 — sanity-check import ngay

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## T12.2 — Dựng split sạch từ manifest

Không tính lại hash/ORB trên Colab (có thể lệch nhẹ theo phiên bản thư viện) —
chỉ **áp** manifest đã commit, nên split ở đây giống hệt split đã kiểm tra ở
máy local. Kết quả mong đợi: train 703 / valid 156 / test 154 ảnh, 0 nhóm
cắt ngang split.

In [ ]:
from pathlib import Path

from src.data.dedup_split import MANIFEST_PATH, OUT_DIR, apply_manifest, read_manifest, summarize

RAW_DIR = Path("data/raw/yoga_v1")
manifest = read_manifest(MANIFEST_PATH)
summarize(manifest)
apply_manifest(RAW_DIR, manifest, OUT_DIR)
CLEAN_YAML = str(OUT_DIR / "data.yaml")
print("data.yaml:", CLEAN_YAML)

## T12.3 — Model cũ trên test gốc: đủ 101 ảnh vs 42 ảnh sạch

Đo model cũ **chỉ** trên ảnh test gốc không trùng với train gốc
(`clean_original_files`). Không dùng test của split mới: split mới có thể
chứa ảnh model cũ đã train.

Số đã đo ở local bằng `models/best.onnx` (cùng model, đang chạy trên PROD):
101 ảnh → mAP@0.5 0.994 / mAP@0.5:0.95 0.767; 42 ảnh sạch → 0.991 / 0.716.
Cell này đo lại bằng `best.pt` trên GPU để có số cùng điều kiện với Giai đoạn 2-3.

In [ ]:
import pandas as pd
from ultralytics import YOLO

from src.data.dedup_split import clean_original_files, write_subset_yaml

OLD_WEIGHTS = "runs/detect/yolov8n_v1_baseline/weights/best.pt"
assert os.path.isfile(OLD_WEIGHTS), f"{OLD_WEIGHTS} chưa có — checkpoint của Giai đoạn 2/3."

clean_test_files = clean_original_files(manifest, "test")
CLEAN_ORIG_TEST_YAML = str(
    write_subset_yaml(RAW_DIR, "test", clean_test_files, Path("data/processed/yoga_v1_orig_clean_test"))
)
print(f"{len(clean_test_files)} ảnh test gốc sạch")

old_rows = []
for label, yaml_path in [
    ("test gốc, đủ 101 ảnh (có rò rỉ)", str(RAW_DIR / "data.yaml")),
    (f"test gốc, {len(clean_test_files)} ảnh sạch", CLEAN_ORIG_TEST_YAML),
]:
    r = YOLO(OLD_WEIGHTS).val(data=yaml_path, split="test", plots=False, verbose=False)
    old_rows.append({"tập": label, "mAP@0.5": r.box.map50, "mAP@0.5:0.95": r.box.map, "P": r.box.mp, "R": r.box.mr})
pd.DataFrame(old_rows).set_index("tập").round(4)

## T12.5 — Train lại baseline trên split sạch

Cùng config chính thức (Giai đoạn 1/3): `yolov8n.pt`, `seed=42`, `epochs=50`,
`imgsz=640`, augmentation ON. Run mới, không ghi đè `yolov8n_v1_baseline`.

In [ ]:
from src.models.train import train

results_clean = train(
    data=CLEAN_YAML,
    model="yolov8n.pt",
    seed=42,
    epochs=50,
    name="yolov8n_v1_clean_baseline",
    exist_ok=True,
)
print("save_dir:", results_clean.save_dir)

Số `results_clean.box` ở trên là trên **valid**. Số báo cáo chính thức phải
trên **test** (`split="test"`) — tập không hề được dùng để chọn checkpoint.

In [ ]:
from src.evaluation.metrics import load_confusion_matrix, top_confused_pairs

CLEAN_WEIGHTS = "runs/detect/yolov8n_v1_clean_baseline/weights/best.pt"
test_clean = YOLO(CLEAN_WEIGHTS).val(data=CLEAN_YAML, split="test", verbose=False)
print(f"TEST mới — mAP@0.5 {test_clean.box.map50:.4f} | mAP@0.5:0.95 {test_clean.box.map:.4f} "
      f"| P {test_clean.box.mp:.4f} | R {test_clean.box.mr:.4f}")
print("per-class mAP@0.5:0.95:", {test_clean.names[i]: round(float(v), 3) for i, v in enumerate(test_clean.box.maps)})

matrix, cm_names = load_confusion_matrix(CLEAN_WEIGHTS, CLEAN_YAML, split="test")
print("Cặp lớp hay nhầm nhất (thật -> đoán: số lần):", top_confused_pairs(matrix, cm_names, k=5) or "không có")
print("Cả background (bỏ sót / phát hiện thừa):", top_confused_pairs(matrix, cm_names, k=5, exclude_background=False))

## T12.6 — Error analysis trên lỗi thật

`per_image_errors()` (`src/evaluation/metrics.py`) liệt kê từng ảnh test có
lỗi: `missed` (bỏ sót), `wrong_class` (nhầm lớp), `spurious` (box thừa).
Dùng `conf=0.5` — đúng ngưỡng app đang dùng trên PROD
(`DETECTOR_CONF_THRESHOLD`), để lỗi ở đây là lỗi người dùng thật thấy.

In [ ]:
import math

import matplotlib.pyplot as plt

from src.evaluation.metrics import per_image_errors

CLASS_NAMES = ["bridge", "downward", "plank", "shoulderstand", "tree"]
clean_model = YOLO(CLEAN_WEIGHTS)
test_errors = per_image_errors(
    clean_model, OUT_DIR / "test" / "images", OUT_DIR / "test" / "labels", CLASS_NAMES, conf=0.5
)
print(f"{len(test_errors)} ảnh có lỗi / {len(list((OUT_DIR / 'test' / 'images').iterdir()))} ảnh test")
for e in test_errors:
    print(f"  {e['path'].name[:40]:40s} {e['errors']}")


def plot_grid(items, title_fn, cols=4):
    if not items:
        print("(không có ảnh nào)")
        return
    rows = math.ceil(len(items) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows), squeeze=False)
    axes = axes.flatten()
    for ax, item in zip(axes, items):
        ax.imshow(item["result"].plot()[:, :, ::-1])
        ax.set_title(title_fn(item), fontsize=8)
        ax.axis("off")
    for ax in axes[len(items):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


plot_grid(test_errors, lambda e: "; ".join(f"{k}: {t}->{p}" for k, t, p, _ in e["errors"]))

### EigenCAM trên ảnh **sai**

Giai đoạn 4 chỉ có ảnh đúng để minh hoạ. Giờ có lỗi thật: xem model nhìn vào
đâu khi sai (nền? người khác? chữ/watermark?). Cùng cách dùng EigenCAM như
`03_evaluation_error_analysis.ipynb` (target layer ngay trước `Detect`).

In [ ]:
import cv2
import numpy as np
from pytorch_grad_cam import EigenCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

torch_model = clean_model.model
torch_model.eval()
cam = EigenCAM(torch_model, [torch_model.model[-2]])


def eigencam_overlay(image_path, imgsz=640):
    rgb = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    rgb_float = np.float32(cv2.resize(rgb, (imgsz, imgsz))) / 255.0
    tensor = torch.from_numpy(rgb_float).permute(2, 0, 1).unsqueeze(0)
    # targets=[] (không phải None) — xem lý do ở notebook 03.
    return show_cam_on_image(rgb_float, cam(tensor, targets=[])[0], use_rgb=True)


cam_samples = test_errors[:5]
if cam_samples:
    fig, axes = plt.subplots(1, len(cam_samples), figsize=(5 * len(cam_samples), 5))
    axes = [axes] if len(cam_samples) == 1 else axes
    for ax, e in zip(axes, cam_samples):
        ax.imshow(eigencam_overlay(e["path"]))
        ax.set_title("; ".join(f"{k}: {t}->{p}" for k, t, p, _ in e["errors"]), fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

### OOD accuracy (ảnh ngoài dataset Roboflow)

Cùng logic với `05_feedback_loop_ood.ipynb` (nhãn lấy từ tên thư mục con hoặc
tiền tố tên file, map qua `CLASS_NAME_MAP`). Tập OOD mở rộng ở T12.4 — số ảnh
hiện có là bao nhiêu thì đo bấy nhiêu.

In [ ]:
OOD_ROOT = Path("data/ood_samples")
CLASS_NAME_MAP = {"downdog": "downward"}


def collect_ood_samples():
    samples = []
    for p in sorted(OOD_ROOT.rglob("*")):
        if p.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        for raw in ([p.parent.name] if p.parent != OOD_ROOT else []) + [p.stem.split("_")[0]]:
            cls = CLASS_NAME_MAP.get(raw, raw)
            if cls in CLASS_NAMES:
                samples.append({"path": p, "true": cls})
                break
        else:
            print("bỏ qua (không suy ra được lớp):", p)
    return samples


def evaluate_ood(model, samples, conf=0.5):
    out = []
    for s in samples:
        result = model.predict(source=str(s["path"]), conf=conf, verbose=False)[0]
        if len(result.boxes):
            top = int(result.boxes.conf.argmax())
            pred, score = CLASS_NAMES[int(result.boxes.cls[top])], float(result.boxes.conf[top])
        else:
            pred, score = None, 0.0
        out.append({**s, "pred": pred, "conf": score, "correct": pred == s["true"], "result": result})
    return out


ood_samples = collect_ood_samples()
print(f"{len(ood_samples)} ảnh OOD, theo lớp:", pd.Series([s["true"] for s in ood_samples]).value_counts().to_dict())
ood_before = evaluate_ood(clean_model, ood_samples)
ood_acc_before = sum(o["correct"] for o in ood_before) / len(ood_before)
print(f"OOD accuracy (clean baseline): {ood_acc_before:.1%}")
ood_wrong = [o for o in ood_before if not o["correct"]]
plot_grid(ood_wrong, lambda o: f"thật={o['true']} đoán={o['pred']} ({o['conf']:.2f})")

### Tóm tắt pattern lỗi

*(Điền tay sau khi xem 2 lưới ảnh + EigenCAM ở trên: lỗi nào nhiều nhất — người
nhỏ/xa, nền rối, ảnh tối, góc chụp lạ, tư thế lộn ngược, ảnh đồ hoạ... Báo lại
cho Claude để ghi vào `docs/problem_statement.md`.)*

## T12.7 — Một vòng cải tiến (rubric mục 5)

Chọn **đúng 1** thay đổi khớp pattern lỗi nhiều nhất ở trên, đặt vào
`OBSERVED_ISSUE`. Bảng tweak giống Giai đoạn 5, thêm 1 lựa chọn đổi model
(`yolov8s.pt`) nếu lỗi không thuộc loại augmentation nào (vd box không khít —
mAP@0.5:0.95 thấp).

**Sửa `OBSERVED_ISSUE` rồi mới chạy cell dưới.**

In [ ]:
TWEAKS = {
    "dark_lighting": {"hsv_v": 0.8},  # mặc định Ultralytics: 0.4
    "small_person": {"scale": 0.9},  # mặc định Ultralytics: 0.5
    "cluttered_background": {"copy_paste": 0.3},  # mosaic mặc định đã 1.0
    "blur_or_odd_angle": {"degrees": 20.0, "shear": 5.0},  # đang: degrees=10, shear=0
    "loose_boxes_bigger_model": {"model": "yolov8s.pt"},  # đổi model thay vì augmentation
}

OBSERVED_ISSUE = None  # <-- đổi thành 1 key ở trên sau khi xem T12.6

assert OBSERVED_ISSUE in TWEAKS, f"Chọn OBSERVED_ISSUE trong {list(TWEAKS)} trước khi retrain."
tweak = dict(TWEAKS[OBSERVED_ISSUE])
base_model = tweak.pop("model", "yolov8n.pt")
print(f"Thay đổi duy nhất: {OBSERVED_ISSUE} -> model={base_model}, {tweak}")

results_feedback = train(
    data=CLEAN_YAML,
    model=base_model,
    seed=42,
    epochs=50,
    name="yolov8n_v1_clean_feedback",
    exist_ok=True,
    **tweak,
)

### Bảng trước/sau — cùng test mới, cùng tập OOD, cùng ngưỡng conf

Ghi đúng kết quả kể cả khi "sau" không tốt hơn.

In [ ]:
FEEDBACK_WEIGHTS = "runs/detect/yolov8n_v1_clean_feedback/weights/best.pt"
feedback_model = YOLO(FEEDBACK_WEIGHTS)
test_feedback = feedback_model.val(data=CLEAN_YAML, split="test", verbose=False)
feedback_errors = per_image_errors(
    feedback_model, OUT_DIR / "test" / "images", OUT_DIR / "test" / "labels", CLASS_NAMES, conf=0.5
)
ood_after = evaluate_ood(feedback_model, ood_samples)
ood_acc_after = sum(o["correct"] for o in ood_after) / len(ood_after)

pd.DataFrame(
    [
        {
            "run": "yolov8n_v1_clean_baseline (trước)",
            "mAP@0.5 (test)": test_clean.box.map50,
            "mAP@0.5:0.95 (test)": test_clean.box.map,
            "ảnh test có lỗi (conf 0.5)": len(test_errors),
            "OOD accuracy": f"{ood_acc_before:.1%}",
        },
        {
            "run": f"yolov8n_v1_clean_feedback (sau, {OBSERVED_ISSUE})",
            "mAP@0.5 (test)": test_feedback.box.map50,
            "mAP@0.5:0.95 (test)": test_feedback.box.map,
            "ảnh test có lỗi (conf 0.5)": len(feedback_errors),
            "OOD accuracy": f"{ood_acc_after:.1%}",
        },
    ]
).set_index("run").round(4)

## Kết luận

*(Điền tay sau khi có số thật — chọn run thắng để export ONNX ở T12.8.)*